In [41]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/featurestorebook/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    if root_dir.parts[-1:] == ('pollen',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Local environment
Added the following directory to the PYTHONPATH: /Users/hongjiang/git/mlfs-book
HopsworksSettings initialized!


In [70]:
import os
import pandas as pd
from datetime import datetime
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import hopsworks
import json
import warnings
warnings.filterwarnings("ignore")

In [71]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store()

secrets = hopsworks.get_secrets_api()
location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)
city = location['city']

2025-12-29 20:39:13,208 INFO: Closing external client and cleaning up certificates.
2025-12-29 20:39:13,211 INFO: Connection closed.
2025-12-29 20:39:13,212 INFO: Initializing external client
2025-12-29 20:39:13,212 INFO: Base URL: https://c.app.hopsworks.ai:443


2025-12-29 20:39:14,552 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1292436


In [78]:
pollen_weather_fg = fs.get_feature_group(name='pollen_weather_training_set', version=1)

In [ ]:
xgb_regressor = XGBRegressor()
xgb_regressor.fit(X_features, y_train)

In [ ]:
y_pred = xgb_regressor.predict(X_test_features)
mse = mean_squared_error(y_test.iloc[:,0], y_pred)
r2 = r2_score(y_test.iloc[:,0], y_pred)
print(f"MSE: {mse}, R2: {r2}")

In [ ]:
from hsml.schema import Schema
from hsml.model_schema import ModelSchema

model_dir = "pollen_model"
os.makedirs(model_dir, exist_ok=True)

xgb_regressor.save_model(f"{model_dir}/model.json")

input_schema = Schema(X_train)
output_schema = Schema(y_train)
model_schema = ModelSchema(input_schema=input_schema, output_schema=output_schema)

mr = project.get_model_registry()
pollen_model = mr.python.create_model(
    name="pollen_xgboost_model",
    metrics={"MSE": mse, "R2": r2},
    model_schema=model_schema,
    feature_view=feature_view,
    description="Pollen Predictor",
)
pollen_model.save(model_dir)